In [1]:
import pandas as pd
import spacy 
from collections import Counter
import matplotlib.pyplot as plt

seed = 1984
nlp = spacy.load("en_core_web_sm")

In [3]:
docs_df = pd.read_csv("./data/documents.csv")
docs_df.sample(5)

,AGORA ID,Official name,Casual name,Link to document,Authority,Collections,Most recent activity,Most recent activity date,Proposed date,Annotated?,...,"Strategies: Licensing, registration, and certification",Strategies: New institution,Strategies: Performance requirements,Strategies: Pilots and testbeds,Strategies: Tiering,Strategies: Tiering: Tiering based on domain of application,Strategies: Tiering: Tiering based on generality,Strategies: Tiering: Tiering based on impact,Strategies: Tiering: Tiering based on inputs,Strategies: Tiering: Tiering based on planning ability
74,1379,Council of Europe Framework Convention on Arti...,Council of Europe Framework Convention on AI,https://rm.coe.int/1680afae3c,Other multinational,Multinational;Editors' Picks,Enacted,2024-05-17,2024-05-17,True,...,False,True,False,True,True,True,False,True,False,False
13,1558,California Senate Bill 942 (2024),California AI Transparency Act (2024),https://leginfo.legislature.ca.gov/faces/billN...,California,U.S. state and local documents,Enacted,2024-09-20,2024-09-20,True,...,True,False,False,False,True,True,False,False,False,False
777,323,Drone Safety and Efficiency Act,Drone Safety and Efficiency Act,https://www.congress.gov/bill/118th-congress/h...,United States Congress,U.S. federal laws,Defunct,2025-01-03,2023-06-09,True,...,False,False,False,False,False,False,False,False,False,False
56,2528,"One Big Beautiful Bill Act 2025, Section 11220...","One Big Beautiful Bill Act 2025, Section 11220...",https://www.congress.gov/bill/119th-congress/h...,United States Congress,U.S. federal laws,Proposed,2025-05-20,2025-05-20,False,...,False,False,False,False,False,False,False,False,False,False
197,1743,Servicemember Quality of Life Improvement and ...,"FY2025 NDAA, Section 1514 (""Management and cyb...",https://www.congress.gov/bill/118th-congress/h...,United States Congress,FY2025 NDAA;NDAA provisions,Enacted,2024-12-23,2024-12-11,True,...,False,False,False,False,False,False,False,False,False,False


In [4]:
# check for proportion of NA in Authority, Collections, Tags
docs_df[['Authority', 'Collections', 'Tags']].isna().sum() /len(docs_df)

Authority      0.000000
Collections    0.002103
Tags           0.555205
dtype: float64

In [5]:
# about 55 % of the tags are missing; 
docs_df.loc[docs_df.Tags.notna(), 'Tags'].sample(10)

406    Harms: Financial loss;Strategies: Government s...
381    Risk factors: Bias;Risk factors: Privacy;Risk ...
536    Risk factors: Transparency;Strategies: Disclos...
342    Strategies: Government study or report;Applica...
674                           Strategies: Input controls
167    Applications: Medicine, life sciences and publ...
678    Strategies: Input controls: Compute circulatio...
21     Strategies: Government support: AI workforce-r...
72     Strategies: Government study or report;Strateg...
177    Strategies: Government support;Strategies: Gov...
Name: Tags, dtype: object

In [6]:
# when Tags is empty are all the subsequent field false ? Yes
# so Tags 55 % of the tags are missing and the individual fields are marked as False.
docs_df.loc[docs_df.Tags.isna(), docs_df.columns[23:]].sum().sum()

np.int64(0)

In [ ]:
# how many long summaries are not usuable ?
docs_df[['Long summary']].isna().sum().values

array([4])

In [ ]:
n_samples = 30
with open(f'prompt_tests/{n_samples}_long_summaries.txt', 'w', encoding='utf-8') as w:
    arr = docs_df[['Long summary']].sample(n_samples, random_state=seed).values
    for i, a in enumerate(arr):
        w.write(f'DOC_ID: {i}\n')
        w.write(f'{a[0]}')
        w.write('\n\n---\n\n')

In [6]:
def plot_topk(tokens, k=20):

    # visualize the frequency of top k most common words
    word_frequency = Counter(tokens)
    topk = word_frequency.most_common(k)
    words, count = zip(*topk)
    plt.barh(words[::-1], count[::-1])
    plt.show()


def spacy_preprocessing(text):
    # tokenization and lemmzer using spaCy; spacy does not have a stemmer
    nlp = spacy.load("en_core_web_sm")
    doc = nlp(text)

    # lemmas without stopwords without punctuations.
    lemmas = [t.lemma_ for t in doc if not t.is_stop and t.is_alpha]
    return lemmas

In [ ]:
# write all the documents in one file

# with open('documents.txt', 'w', encoding='utf-8') as w:
#     for index, row in docs_df.iterrows():
#         if pd.notnull(row['Long summary']):
#             w.write(row['Long summary'])
#             w.write('\n')

In [8]:
tokens = []
for line in open('documents.txt', 'r', encoding='utf-8'):
    tokens.extend(spacy_preprocessing(line))

KeyboardInterrupt: 